In [14]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import statsmodels.api as sm

# Daten einlesen

In [15]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")

# Unterschliedliche Positionen zwiscnen BR und BV

In [16]:
# Nur Abstimmungen wo BR und Parlament unterschiedlicher Meinung sind
dissens = df[df['br-pos'] != df['bv-pos']].dropna(subset=['br-pos', 'bv-pos'])
print(f"Anzahl Fälle wo BR-Pos != BV-Pos: {len(dissens)}")

Anzahl Fälle wo BR-Pos != BV-Pos: 12


In [17]:

dissens = df[df['br-pos'] != df['bv-pos']].dropna(subset=['br-pos', 'bv-pos'])
print(dissens.groupby('hauptgruppe')['titel_kurz_d'].count().sort_values(ascending=False))

hauptgruppe
Staatsordnung                       4
Wirtschaft                          3
Sozial- und Gesellschaftspolitik    2
Umwelt & Lebensraum                 2
Landwirtschaft                      1
Name: titel_kurz_d, dtype: int64


# Modell
Wie stark sollen Ja-Anteil und Stimmbeteiligung in den Zustimmungsfaktor einfliessen?


## Ansatz: Logistische Regression (VERWORFEN)
Zum Bestimmen der Gewichte, fitten wir eine logistische Regression.


In [18]:
temp = df[['volkja-proz', 'beteiligung', 'rechtsform_name']].dropna()
temp = pd.get_dummies(temp, columns=['rechtsform_name'], drop_first=True)
temp = temp.astype(float)

X = sm.add_constant(temp.drop(columns=['volkja-proz']))
y = temp['volkja-proz']

result = sm.OLS(y, X).fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:            volkja-proz   R-squared:                       0.425
Model:                            OLS   Adj. R-squared:                  0.421
Method:                 Least Squares   F-statistic:                     99.90
Date:                Wed, 08 Apr 2026   Prob (F-statistic):           8.58e-79
Time:                        14:48:33   Log-Likelihood:                -2737.7
No. Observations:                 681   AIC:                             5487.
Df Residuals:                     675   BIC:                             5515.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

Die Regression zeigt, dass der Ja-Anteil kaum durch die Mobilisierung beeinflusst wird. Es gibt keine datengetriebenes Gewichtung. Deshalb müssen wir uns ein anderes Konzept überlegen.

# Kongruenz Rechnung anhand eines ungewichteten Prozentsatzes

In [19]:
def kongruenz_prozent(df, pos_col, min_n=10):
    """
    Berechnet, wie viel Prozent der Stimmberechtigten
    im Schnitt der Empfehlung eines Akteurs gefolgt sind.

    Parameter:
    - df: DataFrame mit Abstimmungsdaten
    - pos_col: Spaltenname der Position (z.B. 'p-svp')
    - min_n: Mindestanzahl Abstimmungen für gültiges Ergebnis

    Rückgabe:
    - score: Anteil Stimmberechtigte, die im Schnitt folgten (0 bis 1)
    - n: Anzahl ausgewertete Abstimmungen
    """

    # === SCHRITT 1: Daten vorbereiten ===
    # Nur die drei relevanten Spalten behalten
    temp = df[[pos_col, 'volkja-proz', 'beteiligung']].dropna()

    # Sicherstellen, dass die Position eine Zahl ist
    temp[pos_col] = pd.to_numeric(temp[pos_col], errors='coerce')

    # Nur Abstimmungen behalten, wo eine klare Empfehlung vorliegt
    # 1.0 = Ja-Empfehlung, 2.0 = Nein-Empfehlung
    temp = temp[temp[pos_col].isin([1.0, 2.0])]

    # === SCHRITT 2: Genug Daten vorhanden? ===
    if len(temp) < min_n:
        return np.nan, len(temp)

    # === SCHRITT 3: Werte in Anteile umrechnen ===
    # Beispiel: 65% Ja-Stimmen wird zu 0.65
    ja_anteil = temp['volkja-proz'] / 100

    # Beispiel: 45% Beteiligung wird zu 0.45
    beteiligung = temp['beteiligung'] / 100

    # === SCHRITT 4: Wie viele haben gleich gestimmt wie empfohlen? ===
    # Wenn Partei Ja empfiehlt: der Ja-Anteil zählt
    # Wenn Partei Nein empfiehlt: der Nein-Anteil zählt (= 1 minus Ja-Anteil)
    #
    # Beispiel Ja-Empfehlung:  65% stimmten Ja → zustimmung = 0.65
    # Beispiel Nein-Empfehlung: 65% stimmten Ja → zustimmung = 0.35
    ist_ja_empfehlung = (temp[pos_col] == 1.0)
    zustimmung = np.where(ist_ja_empfehlung, ja_anteil, 1 - ja_anteil)

    # === SCHRITT 5: Auf alle Stimmberechtigten umrechnen ===
    # zustimmung bezieht sich nur auf die Stimmenden
    # Wir wollen aber den Anteil ALLER Stimmberechtigten
    #
    # Beispiel: 65% der Stimmenden sagten Ja, Beteiligung 45%
    # → 0.65 * 0.45 = 0.2925 → 29.25% aller Stimmberechtigten folgten
    gefolgt = zustimmung * beteiligung

    # === SCHRITT 6: Durchschnitt über alle Abstimmungen ===
    score = gefolgt.mean()

    return score, len(temp)

In [20]:
score, n = kongruenz_prozent(df, 'p-svp')
print(f"SVP: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'p-mitte')
print(f"Mitte: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'br-pos')
print(f"Bundesrat: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'p-vcs')
print(f"VCS: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")

SVP: 27.19% der Stimmberechtigten folgten im Schnitt (n=594)
Mitte: 28.20% der Stimmberechtigten folgten im Schnitt (n=43)
Bundesrat: 28.16% der Stimmberechtigten folgten im Schnitt (n=554)
VCS: 21.86% der Stimmberechtigten folgten im Schnitt (n=20)


## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [29]:
def kongruenz_detail(df, pos_cols, min_n=10):
    temp = df.copy()

    for col in pos_cols:
        temp[col] = pd.to_numeric(temp[col], errors='coerce')
        mask = temp[col].isin([1.0, 2.0]) & temp['volkja-proz'].notna()
        ja_anteil = temp['volkja-proz'] / 100
        ist_ja = (temp[col] == 1.0)

        temp[f'zustimmung_{col}'] = np.where(
            mask,
            np.where(ist_ja, ja_anteil, 1 - ja_anteil),
            np.nan
        )

    return temp

In [33]:
result = kongruenz_detail(df, ['p-svp', 'p-mitte'])
print(result)

       anr       datum  legisjahr             rechtsform_name  \
0      1.0  1848-09-12  1848-1851  Obligatorisches Referendum   
1      2.0  1866-01-14  1863-1866  Obligatorisches Referendum   
2      3.0  1866-01-14  1863-1866  Obligatorisches Referendum   
3      4.0  1866-01-14  1863-1866  Obligatorisches Referendum   
4      5.0  1866-01-14  1863-1866  Obligatorisches Referendum   
..     ...         ...        ...                         ...   
703  683.0  2026-03-08  2023-2027             Volksinitiative   
704  684.0  2026-03-08  2023-2027             Volksinitiative   
705  685.0  2026-03-08  2023-2027     Fakultatives Referendum   
706  686.0  2026-06-14  2023-2027             Volksinitiative   
707  687.0  2026-06-14  2023-2027     Fakultatives Referendum   

                                          titel_kurz_d  anzahl  beteiligung  \
0    Bundesverfassung der schweizerischen Eidgenoss...       1          NaN   
1                                     Mass und Gewicht       

In [34]:
print(df['p-svp'].dtype)
print(df['p-svp'].dropna().unique()[:20])
print(df['volkja-proz'].dtype)

float64
[ 2. 66.  1.  5.  9.]
float64


In [35]:
result = kongruenz_detail(df, ['p-svp'])
print(result['zustimmung_p-svp'].dropna().head())

80    0.5019
83    0.5629
84    0.4288
85    0.7136
86    0.6643
Name: zustimmung_p-svp, dtype: float64
